# Cholesky



Attach only bcsstk17.mat using Add Input. MAT_PATH overrides discovery; no downloads are used. Float64 PyTorch on CUDA when available, with CPU fallback. CUDA timings synchronize before and after the call.


In [1]:
import time
from pathlib import Path
import numpy as np
import torch
from scipy.io import loadmat
from scipy.sparse import csr_matrix, issparse

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
print("PyTorch:", torch.__version__, "Device:", device)


def data_path(name, path=None):
    if path is not None:
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    matches = sorted(Path("/kaggle/input").rglob(f"{name}.mat"))
    if not matches:
        p = Path("data") / f"{name}.mat"
        if p.is_file():
            return p
    if len(matches) != 1:
        raise FileNotFoundError(f"Attach {name}.mat or set MAT_PATH; matches: {matches}")
    return matches[0]


def tensor(A, sparse=False):
    if sparse:
        C = csr_matrix(A, dtype=np.float64)
        C.sum_duplicates()
        C.sort_indices()
        return torch.sparse_csr_tensor(torch.as_tensor(C.indptr, dtype=torch.int64, device=device),
                                       torch.as_tensor(C.indices, dtype=torch.int64, device=device),
                                       torch.as_tensor(C.data, dtype=dtype, device=device),
                                       size=C.shape, dtype=dtype, device=device)
    return torch.tensor(A.toarray() if issparse(A) else A, dtype=dtype, device=device)


def mv(A, x):
    return (torch.sparse.mm(A, x[:, None])[:, 0] if A.layout == torch.sparse_csr else A @ x)


def timed(fn, *args):
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    result = fn(*args)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return result, time.perf_counter() - start


def backward_substitution(U, b):
    x = torch.zeros_like(b)
    for i in range(len(b)-1, -1, -1):
        if U[i, i] == 0:
            raise ValueError("Zero pivot")
        x[i] = (b[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x


def lu_solve(L, U, b, p=None):
    c = b.clone() if p is None else b[p].clone()
    for i in range(len(c)):
        c[i] = (c[i] - L[i, :i] @ c[:i]) / L[i, i]
    return backward_substitution(U, c)


def relative_residual(A, x, b):
    return (torch.linalg.vector_norm(mv(A, x)-b) / torch.linalg.vector_norm(b).clamp_min(torch.finfo(dtype).tiny)).item()


def iteration_info(A, x, b, k, tol):
    r = relative_residual(A, x, b)
    return {"iterations": k, "residual": r, "converged": np.isfinite(r) and r <= tol}


PyTorch: 2.11.0+cu130 Device: cuda


In [2]:
@torch.no_grad()
def cholesky(A):
    if not torch.allclose(A, A.T):
        raise ValueError("Require symmetric positive definite A")
    C = A.clone()
    L = torch.zeros_like(C)
    for k in range(len(C)):
        if C[k, k] <= 0:
            raise ValueError("Require positive definite A")
        L[k, k] = C[k, k].sqrt()
        L[k+1:, k] = C[k+1:, k] / L[k, k]
        C[k+1:, k+1:] -= torch.outer(L[k+1:, k], L[k+1:, k])
    return L


## 3×3


In [3]:
A = torch.tensor([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]], dtype=dtype, device=device)
x_true = torch.ones(3, dtype=dtype, device=device)
b = A @ x_true
Z = torch.randn((A.shape[1], 3), dtype=dtype, device=device)
_ = cholesky(A)
result, seconds = timed(cholesky, A)
print("Seconds:", seconds)
L = result
x = lu_solve(L, L.T, b)
print("Factorization probe error:", float(torch.linalg.vector_norm(A @ Z - L @ (L.T @ Z)) / torch.linalg.vector_norm(A @ Z)))
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(torch.linalg.vector_norm(x-x_true) / torch.linalg.vector_norm(x_true)))
assert relative_residual(A, x, b) < 1e-8


Seconds: 0.0009492959999874984
Factorization probe error: 7.549099960382123e-17
Relative residual: 1.1563098121742844e-16
Relative solution error: 1.812986607347358e-16


## SuiteSparse: bcsstk17

Attach only bcsstk17.mat (10,974 × 10,974) using Add Input, or set MAT_PATH.

Dense float64 factorization costs O(n³). Each dense matrix needs about 0.90 GiB; the algorithm also needs working copies and factors.


In [4]:
MATRIX = "bcsstk17"
MAT_PATH = None
A_raw = loadmat(data_path(MATRIX, MAT_PATH))["Problem"][0, 0]["A"].astype(np.float64)
A = tensor(A_raw, sparse=False)
print("Matrix:", MATRIX, "shape:", A.shape)
x_true = torch.ones(A.shape[0], dtype=dtype, device=device)
b = mv(A, x_true)
Z = torch.randn((A.shape[1], 3), dtype=dtype, device=device)


Matrix: bcsstk17 shape: torch.Size([10974, 10974])


In [5]:
result, seconds = timed(cholesky, A)
print("Seconds:", seconds)
L = result
x = lu_solve(L, L.T, b)
print("Factorization probe error:", float(torch.linalg.vector_norm(A @ Z - L @ (L.T @ Z)) / torch.linalg.vector_norm(A @ Z)))
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(torch.linalg.vector_norm(x-x_true) / torch.linalg.vector_norm(x_true)))


Seconds: 78.252585356
Factorization probe error: 6.1895777661076e-16
Relative residual: 2.1228844888019205e-15
Relative solution error: 2.121473371833746e-12
